# **Semana 12: RAG con Pinecone y LangChain (NB2 - Práctico/Ejercicios)**

## **Propósito de la Sesión**
Construir un pipeline completo de Retrieval Augmented Generation (RAG) utilizando Pinecone como base de datos vectorial en la nube, LangChain como orquestador, y un modelo de lenguaje (simulado con OpenAI o Hugging Face). Aprenderemos a crear índices, cargar documentos, dividirlos en fragmentos, generar embeddings y realizar preguntas sobre el contenido.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1. **Crear** un índice gratuito en Pinecone.
2. **Cargar documentos** (PDF o texto) y dividirlos en chunks.
3. **Generar embeddings** de los chunks usando sentence-transformers.
4. **Almacenar** los vectores en Pinecone con sus metadatos.
5. **Realizar búsquedas** semánticas sobre los documentos.
6. **Construir un pipeline RAG** que recupere contexto y genere respuestas.
7. **Comparar** ChromaDB (local) vs Pinecone (nube) en términos de uso y limitaciones.

## **Configuración Inicial**

Instalaremos las librerías necesarias: pinecone-client, langchain, sentence-transformers, y otras utilidades.

In [ ]:
# Instalación de librerías
!pip install pinecone-client langchain langchain-community sentence-transformers pypdf tiktoken --quiet

# Importaciones
import pinecone
from langchain.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Pinecone as LangchainPinecone
from langchain.llms import OpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from getpass import getpass

# Configuración de visualización
sns.set_style("whitegrid")
pd.set_option('display.max_colwidth', 200)

print("Librerías instaladas e importadas correctamente.")

---
## **Parte 1: Creación de Índice Gratuito en Pinecone**

### **¿Qué es Pinecone?**

Pinecone es una base de datos vectorial gestionada en la nube. Ofrece un plan gratuito con 1 índice y hasta 100k vectores de 384 dimensiones. Es ideal para prototipos y proyectos pequeños.

### **Pasos para crear un índice:**

1. **Registro**: Ve a [https://www.pinecone.io/](https://www.pinecone.io/) y crea una cuenta gratuita.
2. **Obtener API Key**: En el dashboard, copia tu API Key.
3. **Crear índice**:
   *   Haz clic en "Create Index".
   *   Nombre: `rag-curso` (o el que prefieras).
   *   Dimensiones: `384` (para all-MiniLM-L6-v2).
   *   Métrica: `cosine` (similitud de coseno).
   *   Tipo de pod: `Starter` (gratuito).
   *   Haz clic en "Create Index".

✅ ¡Ya tienes tu índice vectorial en la nube!

In [ ]:
# Configurar credenciales de Pinecone
print("--- CONEXIÓN A PINECONE ---")
PINECONE_API_KEY = getpass("Ingresa tu API Key de Pinecone: ")
PINECONE_ENV = input("Ingresa tu entorno (ej. gcp-starter): ")

# Inicializar conexión
pinecone.init(api_key=PINECONE_API_KEY, environment=PINECONE_ENV)

# Verificar índices existentes
print("\nÍndices disponibles:")
print(pinecone.list_indexes())

# Nombre del índice (debe coincidir con el que creaste)
index_name = "rag-curso"  # Cambia si usaste otro nombre

# Conectar al índice
if index_name in pinecone.list_indexes():
    index = pinecone.Index(index_name)
    print(f"✅ Conectado al índice '{index_name}'")
    
    # Mostrar estadísticas del índice
    stats = index.describe_index_stats()
    print(f"Estadísticas del índice: {stats}")
else:
    print(f"❌ El índice '{index_name}' no existe. Créalo primero en la consola de Pinecone.")

---
## **Parte 2: Carga y Preparación de Documentos**

Trabajaremos con un documento de ejemplo. Usaremos un artículo de Wikipedia sobre Inteligencia Artificial (descargado como PDF) o un texto de muestra.

### **2.1. Descargar documento de ejemplo**

In [ ]:
# Descargar un documento de ejemplo (artículo sobre IA de Wikipedia)
!wget -O ia_article.pdf https://upload.wikimedia.org/wikipedia/commons/thumb/0/0b/Artificial_intelligence_%28AI%29.pdf

# Alternativa: crear un archivo de texto con contenido relevante
texto_ejemplo = """
# Inteligencia Artificial

La inteligencia artificial (IA) es la simulación de procesos de inteligencia humana por parte de máquinas, especialmente sistemas informáticos. Estos procesos incluyen el aprendizaje (la adquisición de información y reglas para usar la información), el razonamiento (usar reglas para llegar a conclusiones aproximadas o definitivas) y la autocorrección.

## Aprendizaje Automático
El aprendizaje automático (machine learning) es una rama de la IA que permite a las máquinas aprender de datos sin ser explícitamente programadas. Los algoritmos de aprendizaje automático construyen un modelo basado en datos de entrenamiento para hacer predicciones o decisiones.

## Aprendizaje Profundo
El aprendizaje profundo (deep learning) es un subcampo del machine learning basado en redes neuronales artificiales con múltiples capas. Ha permitido avances significativos en visión por computadora, procesamiento de lenguaje natural y otras áreas.

## Procesamiento de Lenguaje Natural
El procesamiento de lenguaje natural (PLN) es el campo de la IA que se ocupa de la interacción entre computadoras y humanos mediante el lenguaje natural. Aplicaciones incluyen traducción automática, análisis de sentimientos y chatbots.

## Aplicaciones de la IA
La IA se aplica en diversos sectores: salud (diagnóstico de enfermedades), finanzas (detección de fraude), transporte (vehículos autónomos), educación (tutores inteligentes) y muchos más.
"""

with open('ia_introduccion.txt', 'w') as f:
    f.write(texto_ejemplo)

print("Documento de ejemplo creado: ia_introduccion.txt")

### **2.2. Cargar y dividir documentos en chunks**

Usaremos LangChain para cargar el documento y dividirlo en fragmentos (chunks) de tamaño adecuado.

In [ ]:
# Cargar documento de texto
loader = TextLoader('ia_introduccion.txt')
documentos = loader.load()

# Configurar divisor de texto
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,        # Tamaño de cada fragmento (en caracteres)
    chunk_overlap=50,      # Superposición entre fragmentos
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

# Dividir documentos
chunks = text_splitter.split_documents(documentos)
print(f"Documento original dividido en {len(chunks)} fragmentos.")

# Mostrar algunos chunks
print("\nEjemplos de chunks:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content)
    print(f"Metadatos: {chunk.metadata}")

---
## **Parte 3: Generación de Embeddings y Subida a Pinecone**

Utilizaremos `sentence-transformers` a través de LangChain para generar embeddings de cada chunk y subirlos a Pinecone.

In [ ]:
# Configurar embeddings con Hugging Face
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

print(f"Modelo de embeddings cargado. Dimensión: 384")

In [ ]:
# Subir documentos a Pinecone usando LangChain
print("Subiendo documentos a Pinecone...")
start_time = time.time()

vectorstore = LangchainPinecone.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name
)

upload_time = time.time() - start_time
print(f"✅ Subida completada en {upload_time:.2f} segundos.")

# Verificar estadísticas del índice después de la subida
stats = index.describe_index_stats()
print(f"\nEstadísticas actualizadas:")
print(f"  Total de vectores: {stats['total_vector_count']}")
print(f"  Dimensiones: {stats['dimension']}")

---
## **Parte 4: Búsqueda Semántica en Pinecone**

Antes de construir el RAG completo, probemos la búsqueda semántica directamente.

In [ ]:
def buscar_semantico(query, k=3):
    """Busca fragmentos similares a la consulta"""
    print(f"\n🔍 Consulta: '{query}'")
    
    # Generar embedding de la consulta
    query_emb = embeddings.embed_query(query)
    
    # Buscar en Pinecone
    results = index.query(
        vector=query_emb,
        top_k=k,
        include_metadata=True
    )
    
    print(f"\nResultados:")
    for i, match in enumerate(results['matches']):
        print(f"\n  {i+1}. [Score: {match['score']:.4f}]")
        print(f"     {match['metadata']['text'][:150]}...")
    
    return results

# Probar búsquedas
buscar_semantico("¿Qué es el deep learning?")
buscar_semantico("Aplicaciones de la IA en salud")
buscar_semantico("Cómo funciona el procesamiento de lenguaje natural")

---
## **Parte 5: Pipeline RAG con LangChain**

Ahora construiremos el sistema completo de preguntas y respuestas. Para el LLM, usaremos OpenAI (requiere API key) o una alternativa gratuita como Hugging Face Hub.

### **5.1. Configurar el LLM (OpenAI)**

In [ ]:
# Configurar OpenAI (requiere API key)
OPENAI_API_KEY = getpass("Ingresa tu API Key de OpenAI (opcional): ")

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    from langchain.llms import OpenAI
    llm = OpenAI(temperature=0, model_name="gpt-3.5-turbo-instruct")
    print("✅ LLM de OpenAI configurado.")
else:
    print("⚠️  No se proporcionó API key. Usaremos una simulación.")
    # Simulación simple para demostración
    from langchain.llms import FakeListLLM
    responses = ["La inteligencia artificial es un campo que...", 
                 "El deep learning utiliza redes neuronales...",
                 "Las aplicaciones de IA incluyen salud y finanzas..."]
    llm = FakeListLLM(responses=responses)
    print("✅ Usando LLM simulado.")

### **5.2. Crear el chain de RetrievalQA**

In [ ]:
# Crear el retriever desde Pinecone
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}  # Número de fragmentos a recuperar
)

# Crear prompt personalizado
prompt_template = """Responde la pregunta basándote ÚNICAMENTE en el siguiente contexto. Si no encuentras la respuesta en el contexto, di "No tengo suficiente información para responder."

Contexto:
{context}

Pregunta: {question}
Respuesta:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Crear cadena de QA
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # "stuff", "map_reduce", "refine", "map_rerank"
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True
)

print("✅ Cadena RAG configurada.")

### **5.3. Hacer preguntas al sistema RAG**

In [ ]:
def preguntar(question):
    """Función para hacer preguntas al sistema RAG"""
    print("\n" + "="*70)
    print(f"❓ PREGUNTA: {question}")
    print("="*70)
    
    # Ejecutar la consulta
    result = qa_chain({"query": question})
    
    print("\n📝 RESPUESTA:")
    print(result['result'])
    
    print("\n📚 FUENTES UTILIZADAS:")
    for i, doc in enumerate(result['source_documents']):
        print(f"  {i+1}. {doc.page_content[:100]}...")
    
    return result

# Probar con preguntas sobre el documento
preguntar("¿Qué es el aprendizaje automático?")
preguntar("¿Cómo se relaciona el deep learning con la IA?")
preguntar("¿En qué sectores se aplica la IA?")

---
## **Parte 6: Comparativa ChromaDB vs Pinecone**

Ahora que hemos trabajado con Pinecone, comparemos con ChromaDB (visto en NB1) en varios aspectos.

In [ ]:
comparativa = pd.DataFrame({
    'Característica': ['Tipo', 'Despliegue', 'Costo', 'Límites', 'Facilidad de uso', 'Persistencia', 'Índices soportados', 'Integración con LangChain', 'Ideal para'],
    'ChromaDB': ['Local (embebida)', 'Misma máquina', 'Gratuito', 'Memoria/disco local', 'Muy fácil', 'Sí (archivos)', 'HNSW (configurable)', 'Sí', 'Prototipos, desarrollo local'],
    'Pinecone': ['Nube (gestionado)', 'SaaS', 'Gratuito limitado', '1 índice, 100k vectores', 'Fácil (API)', 'Sí (automática)', 'HNSW (optimizado)', 'Sí', 'Producción, escalabilidad']
})

print("--- COMPARATIVA CHROMADB VS PINECONE ---")
display(comparativa)

print("\n📌 Conclusiones:")
print("• ChromaDB es ideal para desarrollo local, pruebas y cuando los datos no deben salir de la máquina.")
print("• Pinecone es ideal para producción, cuando se necesita escalabilidad y no se quiere gestionar infraestructura.")
print("• Ambos se integran bien con LangChain, lo que permite cambiar entre ellos fácilmente.")

---
## **Ejercicios Prácticos para el Estudiante**

Ahora aplica lo aprendido con estos ejercicios.

### **Ejercicio 1: Indexar un documento propio**

Elige un documento (PDF o texto) de tu interés. Puede ser un artículo, un capítulo de libro, o incluso una página web descargada. Cárgalo, divídelo en chunks (tamaño 300, solapamiento 50) e índexalo en Pinecone. Luego, realiza al menos 3 preguntas sobre su contenido.

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# 1. Cargar documento (puedes usar PyPDFLoader para PDF)
# from langchain.document_loaders import PyPDFLoader
# loader = PyPDFLoader("tu_archivo.pdf")
# ...

# 2. Dividir en chunks
# ...

# 3. Indexar en Pinecone (crear nuevo índice o usar el mismo con namespace diferente)
# ...

# 4. Hacer preguntas
# ...

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 2: Experimentar con diferentes tamaños de chunk**

Prueba al menos 3 tamaños de chunk diferentes (ej. 100, 300, 500) con el mismo documento. Para cada tamaño:
1. Indexa los documentos en Pinecone (puedes usar namespaces diferentes: `chunk_100`, `chunk_300`, etc.)
2. Haz las mismas 3 preguntas a cada configuración.
3. Compara la calidad de las respuestas. ¿Qué tamaño funciona mejor? ¿Por qué?

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# Tamaños a probar
chunk_sizes = [100, 300, 500]

for size in chunk_sizes:
    print(f"\n{'='*50}")
    print(f"Probando chunk_size = {size}")
    print('='*50)
    # ...

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 3: Añadir metadatos a los documentos**

Modifica el proceso de indexación para incluir metadatos adicionales en cada chunk, como:
*   `fuente`: nombre del documento original.
*   `página`: número de página (si aplica).
*   `categoría`: tema del chunk (puedes asignarlo manualmente o inferirlo).

Luego, realiza una búsqueda que filtre por metadatos (por ejemplo, buscar solo en chunks de una categoría específica).

In [ ]:
# --- INICIO DE TU CÓDIGO ---

# Crear documentos con metadatos
# ...

# Indexar
# ...

# Buscar con filtro (Pinecone soporta filtros en metadata)
# index.query(vector=query_emb, filter={"categoria": "machine_learning"}, top_k=3)

# --- FIN DE TU CÓDIGO ---

### **Ejercicio 4: Reflexión sobre RAG en producción**

En una celda Markdown, responde:
1. ¿Qué ventajas ofrece Pinecone sobre ChromaDB para un sistema en producción?
2. ¿Qué consideraciones de costo y latencia deberías tener en cuenta?
3. ¿Cómo manejarías la actualización de documentos (añadir, modificar, eliminar) en un sistema RAG basado en Pinecone?
4. Investiga: ¿Qué otros servicios de bases de datos vectoriales existen en la nube (AWS, GCP, Azure)?

---
## **Conclusiones**

En esta sesión práctica hemos:
1. **Configurado** un índice gratuito en Pinecone.
2. **Cargado y procesado** documentos, dividiéndolos en chunks.
3. **Generado embeddings** y subido los vectores a Pinecone.
4. **Construido** un pipeline RAG completo con LangChain.
5. **Comparado** ChromaDB (local) vs Pinecone (nube).

**Conceptos clave reforzados:**
*   RAG combina recuperación (búsqueda semántica) y generación (LLM).
*   Las bases de datos vectoriales en la nube como Pinecone ofrecen escalabilidad sin gestión de infraestructura.
*   LangChain abstrae las complejidades de la integración entre componentes.
*   La elección entre solución local y en la nube depende de requisitos de escala, costo y privacidad.

¡Has construido tu primer sistema RAG funcional!

In [ ]:
# (Opcional) Limpiar recursos si es necesario
# No eliminamos el índice para no perder el progreso, pero en producción podrías hacerlo.
# pinecone.delete_index(index_name)
print("\nNota: El índice en Pinecone permanece para futuros usos.")